In [1]:
# %pip install -U langchain-groq langchain-core langsmith python-dotenv pydantic

import os
import warnings
import logging
import contextlib
import io
from collections import defaultdict
from statistics import mean
from time import perf_counter

from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client, evaluate
from pydantic import BaseModel, Field

DELETE_DATASET = False
CREATE_DATASET = False

warnings.filterwarnings("ignore")
logging.getLogger("langsmith").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

load_dotenv(find_dotenv())

print("GROQ_API_KEY loaded:", bool(os.getenv("GROQ_API_KEY")))
print("LANGSMITH_API_KEY loaded:", bool(os.getenv("LANGSMITH_API_KEY")))

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = ("exercise-1-groq-comparison")

C:\Users\User\anaconda3\envs\ai-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GROQ_API_KEY loaded: True
LANGSMITH_API_KEY loaded: True


In [2]:
prompt = ChatPromptTemplate.from_template("""Answer the question using only the supplied context.

Question:
{question}

Context:
{context}""")

question = ("What is one of the best places to see orcas "
            "in the United States?")

context = """The San Juan Islands in Washington State are one of the
best-known places in the United States for observing orcas.
Whale-watching tours commonly depart from Friday Harbor,
although wildlife sightings are never guaranteed."""

reference_answer = ("The San Juan Islands in Washington State are one of "
                    "the best places in the United States to see orcas.")

In [3]:
models = ["llama-3.1-8b-instant",
          "llama-3.3-70b-versatile",
          "openai/gpt-oss-20b",
          "openai/gpt-oss-120b",]

In [4]:
client = Client()

def delete_dataset_if_exists(client: Client,
                             dataset_name: str) -> bool:
    """Delete a LangSmith dataset if it exists."""

    datasets = list(client.list_datasets(dataset_name=dataset_name))
    if not datasets:
        print(f"Dataset does not exist: {dataset_name}")
        return False

    for dataset in datasets:
        client.delete_dataset(dataset_id=dataset.id)
    print(f"Dataset deleted: {dataset_name}")
    return True


def create_model_comparison_dataset(client: Client,
                                    dataset_name: str,
                                    models: list[str],
                                    question: str,
                                    context: str,
                                    reference_answer: str):
    """Create a dataset for comparing multiple models."""

    if client.has_dataset(dataset_name=dataset_name):
        print(f"Dataset already exists: {dataset_name}")
        return None

    dataset = client.create_dataset(dataset_name=dataset_name,
                                    description=(
            "Compares multiple Groq models using the same "
            "question, context, and reference answer."))

    examples = []

    for model_name in models:
        examples.append({"inputs": {"model_name": model_name,
                                    "question": question,
                                    "context": context},
                         "outputs": {"reference_answer": reference_answer}})

    client.create_examples(dataset_id=dataset.id,
                           examples=examples)

    print(f"Dataset created: {dataset_name}")
    return dataset

In [5]:
dataset_name = "exercise-1-groq-model-comparison-v2"

if DELETE_DATASET:
    delete_dataset_if_exists(client=client,
                             dataset_name=dataset_name)

if CREATE_DATASET:
    create_model_comparison_dataset(client=client,
                                    dataset_name=dataset_name,
                                    models=models,
                                    question=question,
                                    context=context,
                                    reference_answer=reference_answer)

else:
    if client.has_dataset(dataset_name=dataset_name):
        print("Using existing LangSmith dataset:", dataset_name)
    else:
        raise ValueError(f"Dataset '{dataset_name}' does not exist. "
                         "Set CREATE_DATASET = True to create it.")

Using existing LangSmith dataset: exercise-1-groq-model-comparison-v2


In [6]:
performance_results = []

def run_groq_model(inputs: dict) -> dict:
    """Run one Groq model and collect performance data."""

    model_name = inputs["model_name"]
    model = ChatGroq(model=model_name, temperature=0)
    chain = prompt | model
    start_time = perf_counter()
    
    response = chain.invoke({"question": inputs["question"],
                             "context": inputs["context"]})

    response_time = perf_counter() - start_time
    usage = response.usage_metadata or {}
    input_tokens = usage.get("input_tokens", 0)
    output_tokens = usage.get("output_tokens", 0)
    total_tokens = usage.get("total_tokens", 0)

    # Save the result for the local performance summary.
    performance_results.append({"model_name": model_name,
                                "response_time_seconds": response_time,
                                "input_tokens": input_tokens,
                                "output_tokens": output_tokens,
                                "total_tokens": total_tokens})

    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("RESPONSE TIME:", f"{response_time:.4f} seconds")
    print("INPUT TOKENS:", input_tokens)
    print("OUTPUT TOKENS:", output_tokens)
    print("TOTAL TOKENS:", total_tokens)
    print("\nRESPONSE CONTENT:")
    print(response.content)

    return {"model_name": model_name,
            "response": response.content,
            "response_time_seconds": response_time,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens}

In [7]:
class QualityEvaluation(BaseModel):
    """Structured result returned by the evaluator model."""

    score: float = Field(ge=0, le=1, description=(
            "Overall response-quality score from 0 to 1."))
    reasoning: str = Field(
        description=("A brief explanation of the assigned score."))

evaluation_prompt = ChatPromptTemplate.from_template("""
You are evaluating the quality of an AI-generated response.

Evaluate the response according to these criteria:

1. Correctness:
   Does it agree with the reference answer?

2. Relevance:
   Does it directly answer the user's question?

3. Grounding:
   Is it supported by the supplied context?

4. Completeness:
   Does it include the important information needed to
   answer the question?

Assign an overall score between 0 and 1:
- 1.0 means fully correct, relevant, grounded, and complete.
- 0.75 means mostly correct with a minor weakness.
- 0.5 means partially correct or incomplete.
- 0.25 means mostly incorrect or poorly supported.
- 0.0 means completely incorrect or irrelevant.

Question:
{question}

Context:
{context}

Reference answer:
{reference_answer}

Generated response:
{generated_answer}""")

evaluation_model = ChatGroq(model="llama-3.1-8b-instant",
                            temperature=0
).with_structured_output(QualityEvaluation)

evaluation_chain = (evaluation_prompt | evaluation_model)

def evaluate_response_quality(inputs: dict,
                              outputs: dict,
                              reference_outputs: dict) -> dict:
    """Evaluate a response against its reference answer."""

    generated_answer = outputs.get("response", "")
    reference_answer = reference_outputs.get("reference_answer", "")

    if not generated_answer.strip():
        return {"key": "response_quality",
                "score": 0.0,
                "comment": "The model returned an empty response."}

    if not reference_answer.strip():
        return {"key": "response_quality",
                "score": 0.0,
                "comment": ("The dataset does not contain a "
                            "reference answer.")}

    quality_evaluation = evaluation_chain.invoke({
            "question": inputs.get("question",""),
            "context": inputs.get("context", ""),
            "reference_answer": reference_answer,
            "generated_answer": generated_answer})

    return {"key": "response_quality",
            "score": quality_evaluation.score,
            "comment": quality_evaluation.reasoning}

In [8]:
def display_performance_summary(results: list[dict]) -> None:
    """Display average performance for every tested model."""

    if not results:
        print("No performance results were recorded.")
        return

    grouped_results = defaultdict(list)

    for result in results:
        grouped_results[result["model_name"]].append(result)

    summary = []

    for model_name, model_runs in grouped_results.items():
        average_time = mean(run["response_time_seconds"] for run in model_runs)
        average_input_tokens = mean(run["input_tokens"] for run in model_runs)
        average_output_tokens = mean(run["output_tokens"] for run in model_runs)
        average_total_tokens = mean(run["total_tokens"] for run in model_runs)
        summary.append({"model_name": model_name,
                        "average_time": average_time,
                        "average_input_tokens": average_input_tokens,
                        "average_output_tokens": average_output_tokens,
                        "average_total_tokens": average_total_tokens})
        
    summary.sort(key=lambda item: item["average_time"])

    print("\n" + "=" * 70)
    print("MODEL PERFORMANCE SUMMARY")
    print("=" * 70)
    for position, result in enumerate(summary, start=1):
        print(f"\n{position}. {result['model_name']}")
        print("   Average response time:", f"{result['average_time']:.4f} seconds")
        print("   Average input tokens:", f"{result['average_input_tokens']:.1f}") 
        print("   Average output tokens:", f"{result['average_output_tokens']:.1f}")
        print("   Average total tokens:", f"{result['average_total_tokens']:.1f}")

    print("\nFastest model:", summary[0]["model_name"])

In [9]:
_stderr_buffer = io.StringIO()
with contextlib.redirect_stderr(_stderr_buffer):
    evaluation_results = evaluate(run_groq_model,
                                  data=dataset_name,
                                  evaluators=[evaluate_response_quality],
                                  experiment_prefix="exercise-1-groq-comparison",
                                  num_repetitions=2,
                                  max_concurrency=1)

View the evaluation results for experiment: 'exercise-1-groq-comparison-4570c71e' at:
https://smith.langchain.com/o/45a596ad-60b3-4c9d-9bf9-b755074c57ec/datasets/ef0f768d-d179-4d20-8b5a-484f0904dd34/compare?selectedSessions=ed0ae99e-b0b0-4840-a6b5-07191fff6f98



MODEL: openai/gpt-oss-120b
RESPONSE TIME: 0.8814 seconds
INPUT TOKENS: 142
OUTPUT TOKENS: 90
TOTAL TOKENS: 232

RESPONSE CONTENT:
The San Juan Islands in Washington State.

MODEL: openai/gpt-oss-20b
RESPONSE TIME: 0.8615 seconds
INPUT TOKENS: 142
OUTPUT TOKENS: 106
TOTAL TOKENS: 248

RESPONSE CONTENT:
The San Juan Islands in Washington State.

MODEL: openai/gpt-oss-120b
RESPONSE TIME: 1.0662 seconds
INPUT TOKENS: 142
OUTPUT TOKENS: 90
TOTAL TOKENS: 232

RESPONSE CONTENT:
The San Juan Islands in Washington State.

MODEL: openai/gpt-oss-20b
RESPONSE TIME: 0.6215 seconds
INPUT TOKENS: 142
OUTPUT TOKENS: 106
TOTAL TOKENS: 248

RESPONSE CONTENT:
The San Juan Islands in Washington State.


In [10]:
display_performance_summary(performance_results)


MODEL PERFORMANCE SUMMARY

1. openai/gpt-oss-20b
   Average response time: 0.7415 seconds
   Average input tokens: 142.0
   Average output tokens: 106.0
   Average total tokens: 248.0

2. openai/gpt-oss-120b
   Average response time: 0.9738 seconds
   Average input tokens: 142.0
   Average output tokens: 90.0
   Average total tokens: 232.0

Fastest model: openai/gpt-oss-20b
